# 实验二参考答案：科研级数据可视化

> 对应讲次：第2讲 数据高阶可视化 × AI 辅助表达
>
> 数据集：global_temperature.csv（NASA GISTEMP v4，1880-2024，Source/Year/Month/Mean）
>
> AI 角色：设计顾问 — 图表方案推荐 / 视觉审查 / 三轮迭代

## 环境配置与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 发表级全局配置
plt.rcParams.update({
    'font.sans-serif': ['Arial Unicode MS', 'SimHei', 'DejaVu Sans'],
    'axes.unicode_minus': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Wong 2011 色盲友好配色
WONG = {
    'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73',
    'pink': '#CC79A7', 'lightblue': '#56B4E9', 'red': '#D55E00', 'yellow': '#F0E442'
}

In [ ]:
# 加载数据
df = pd.read_csv('global_temperature.csv')
print(f"数据形状: {df.shape}")
print(f"列: {list(df.columns)}")
print(f"数据源: {df['Source'].unique()}")
print(f"年份: {df['Year'].min()}-{df['Year'].max()}")
print(f"月份: {sorted(df['Month'].unique())}")
print()
print(df.head())

In [ ]:
# 数据预处理
gistemp = df[df['Source'] == 'GISTEMP'].copy()
monthly = gistemp.copy()

# 年度均值
annual = monthly.groupby('Year')['Mean'].mean()
print(f"年度数据: {len(annual)} 年")
print(f"距平范围: [{annual.min():.3f}, {annual.max():.3f}] C")

# 季节划分
season_map = {12: 'DJF', 1: 'DJF', 2: 'DJF', 3: 'MAM', 4: 'MAM', 5: 'MAM',
              6: 'JJA', 7: 'JJA', 8: 'JJA', 9: 'SON', 10: 'SON', 11: 'SON'}
monthly['Season'] = monthly['Month'].map(season_map)
monthly['Decade'] = (monthly['Year'] // 10 * 10).astype(str) + 's'

## 任务1：Matplotlib 基础可视化（20分）

### 1.1 趋势折线图（正负距平分色填充）

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(annual.index, annual.values, color='#333', linewidth=0.8, alpha=0.8)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.fill_between(annual.index, annual.values, 0,
                where=(annual.values > 0), color='#E74C3C', alpha=0.4, label='正距平(变暖)')
ax.fill_between(annual.index, annual.values, 0,
                where=(annual.values <= 0), color='#3498DB', alpha=0.4, label='负距平(偏冷)')

ax.set_title('全球年度气温距平趋势（1880-2024）')
ax.set_xlabel('年份')
ax.set_ylabel('气温距平 (\u2103)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('output/task1_trend.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.2 年代均值柱状图

In [ ]:
decades = annual.groupby(annual.index // 10 * 10).mean()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#3498DB' if v < 0 else '#E74C3C' for v in decades.values]
bars = ax.bar(decades.index.astype(str), decades.values, color=colors, width=0.7, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)

for bar, v in zip(bars, decades.values):
    y_pos = v + 0.02 if v >= 0 else v - 0.05
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{v:.2f}', ha='center', fontsize=9)

ax.set_title('各年代全球平均气温距平')
ax.set_xlabel('年代')
ax.set_ylabel('温度距平 (\u2103)')
plt.tight_layout()
plt.savefig('output/task1_decade.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.3 四季趋势多子图（2x2）

In [ ]:
seasonal_annual = monthly.groupby(['Year', 'Season'])['Mean'].mean().reset_index()
seasons_order = ['MAM', 'JJA', 'SON', 'DJF']
season_labels = {'MAM': '春季 MAM', 'JJA': '夏季 JJA', 'SON': '秋季 SON', 'DJF': '冬季 DJF'}
season_colors = {'MAM': '#27AE60', 'JJA': '#E74C3C', 'SON': '#F39C12', 'DJF': '#3498DB'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, season in zip(axes.flat, seasons_order):
    data = seasonal_annual[seasonal_annual['Season'] == season]
    ax.plot(data['Year'], data['Mean'], color=season_colors[season], linewidth=1, alpha=0.7)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
    ax.set_title(season_labels[season])
    ax.set_ylabel('距平 (\u2103)')

axes[1, 0].set_xlabel('年份')
axes[1, 1].set_xlabel('年份')
plt.suptitle('四季年均气温距平趋势', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('output/task1_seasons.png', dpi=300, bbox_inches='tight')
plt.show()

## 任务2：Seaborn 高阶探索（25分）

### 2.1 热力图：月份 × 年代

In [ ]:
pivot = monthly.pivot_table(values='Mean', index='Month', columns='Decade', aggfunc='mean')
# 按年代排序列
decade_order = sorted(pivot.columns, key=lambda x: int(x[:-1]))
pivot = pivot[decade_order]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
            linewidths=0.5, ax=ax, cbar_kws={'label': '温度距平 (\u2103)'})
ax.set_title('各月份 \u00d7 年代 平均气温距平热力图')
ax.set_xlabel('年代')
ax.set_ylabel('月份')
ax.set_yticklabels([f'{m}月' for m in range(1, 13)], rotation=0)
plt.tight_layout()
plt.savefig('output/task2_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.2 小提琴图：两个时期对比

In [ ]:
monthly['Period'] = monthly['Year'].apply(
    lambda y: '早期(1880-1960)' if y <= 1960 else '近期(1961-2024)')

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=monthly, x='Period', y='Mean',
               palette=[WONG['blue'], WONG['red']], inner='quartile', ax=ax)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('早期 vs 近期 月度气温距平分布对比')
ax.set_xlabel('')
ax.set_ylabel('温度距平 (\u2103)')
plt.tight_layout()
plt.savefig('output/task2_violin.png', dpi=300, bbox_inches='tight')
plt.show()

# 数值对比
for period in ['早期(1880-1960)', '近期(1961-2024)']:
    data = monthly[monthly['Period'] == period]['Mean']
    print(f"{period}: mean={data.mean():.3f}, median={data.median():.3f}, std={data.std():.3f}")

### 2.3 FacetGrid 分面图：四季趋势+回归线

In [ ]:
g = sns.FacetGrid(seasonal_annual, col='Season', col_wrap=2,
                  height=4, aspect=1.5, col_order=seasons_order)
g.map_dataframe(sns.regplot, x='Year', y='Mean',
                scatter_kws={'s': 8, 'alpha': 0.3, 'color': '#666'},
                line_kws={'color': WONG['red'], 'linewidth': 2})
g.set_titles('{col_name}')
g.set_axis_labels('年份', '气温距平 (\u2103)')

for ax in g.axes.flat:
    ax.axhline(0, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('output/task2_facet.png', dpi=300, bbox_inches='tight')
plt.show()

## 任务3：科研论文级图表（30分）

### 3.1 主图：趋势 + 10年滑动平均 + 95% CI

In [ ]:
window = 10
rolling_mean = annual.rolling(window).mean()
rolling_std = annual.rolling(window).std()
ci_upper = rolling_mean + 1.96 * rolling_std / np.sqrt(window)
ci_lower = rolling_mean - 1.96 * rolling_std / np.sqrt(window)

### 3.2 统计标注：t 检验

In [ ]:
# t 检验
early = annual[(annual.index >= 1880) & (annual.index <= 1920)].values
recent = annual[(annual.index >= 1990) & (annual.index <= 2024)].values
t_stat, p_value = stats.ttest_ind(recent, early)

print("=== 独立样本 t 检验 ===")
print(f"早期(1880-1920): n={len(early)}, mean={early.mean():.4f}, std={early.std():.4f}")
print(f"近期(1990-2024): n={len(recent)}, mean={recent.mean():.4f}, std={recent.std():.4f}")
print(f"t = {t_stat:.4f}, p = {p_value:.2e}")
sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
print(f"显著性: {sig} (p < 0.001)")

### 3.3 发表级完整图表（Nature 标准）

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))

# 年度原始值
ax.plot(annual.index, annual.values, color=WONG['blue'], linewidth=0.5, alpha=0.4)

# 10年滑动平均
ax.plot(annual.index, rolling_mean, color=WONG['red'], linewidth=1.5, label='10-yr mean')

# 95% 置信区间
ax.fill_between(annual.index, ci_lower, ci_upper, alpha=0.15, color=WONG['blue'])

# 零线
ax.axhline(0, color='gray', linestyle='--', linewidth=0.4, alpha=0.5)

# 标注最暖年
for yr in [2016, 2023]:
    if yr in annual.index:
        val = annual[yr]
        ax.annotate(str(yr), xy=(yr, val), xytext=(yr-18, val+0.12),
                    fontsize=6.5, ha='center',
                    arrowprops=dict(arrowstyle='->', color='black', lw=0.6))

# 统计标注（方括号+p值）
y_bar = 1.15
ax.plot([1900, 1900, 2007, 2007], [y_bar-0.03, y_bar, y_bar, y_bar-0.03],
        'k-', linewidth=0.6)
ax.text(1953, y_bar + 0.03, f'{sig} p={p_value:.1e}', ha='center', fontsize=6)

# 两段均值虚线
ax.axhline(early.mean(), xmin=0.01, xmax=0.28, color=WONG['blue'],
           linestyle=':', linewidth=0.8, alpha=0.6)
ax.axhline(recent.mean(), xmin=0.76, xmax=0.99, color=WONG['red'],
           linestyle=':', linewidth=0.8, alpha=0.6)

# 排版规范
ax.set_xlabel('Year', fontsize=8)
ax.set_ylabel('Temperature anomaly (\u2103)', fontsize=8)
ax.tick_params(labelsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=6.5, loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig('output/fig_nature.pdf', dpi=300, bbox_inches='tight')
plt.savefig('output/fig_nature.png', dpi=300, bbox_inches='tight')
plt.show()
print("Nature 标准图表已保存: output/fig_nature.pdf + .png")
print(f"尺寸: 3.5 x 2.8 inch | 300 dpi | Wong 2011 配色 | Arial")

### 发表级排版自查

| 检查项 | 要求 | 状态 |
|--------|------|:----:|
| 图片宽度 | 3.5 inch | OK |
| 分辨率 | 300 dpi | OK |
| 背景 | 白底无网格 | OK |
| 字体 | Arial (系统回退) | OK |
| 轴标签字号 | 8pt | OK |
| 配色 | Wong 2011 色盲友好 | OK |
| 坐标轴 | 左底可见，右顶隐藏 | OK |
| 输出格式 | PDF + PNG | OK |

## 任务4：AI 辅助三轮迭代（25分）

### 目标图表选择

选择任务3的主图（趋势+滑动平均+CI+标注），因其复杂度最高，适合展示三轮迭代的递进效果。

### Round 1 — 氛围提示

```
帮我用全球气温数据画一张展示变暖趋势的折线图，要有科学感，
希望能看出长期变暖的加速。
```

**预期 AI 输出**：一张基础折线图，可能有滑动平均线，但缺少置信区间、标注、发表级排版。

**评价**：
- 优点：正确识别了"折线图+滑动平均"的需求
- 不足：默认 dpi 不够、无标注、配色未考虑色盲、字号偏大

### Round 2 — 约束提示

```
保持趋势图结构，按以下要求优化：
1. 加 95% 置信区间带（fill_between, alpha=0.15）
2. 标注 2016 和 2023 为历史最暖年（箭头标注）
3. 加统计标注：1880-1920 vs 1990-2024 t检验结果
4. 配色用 Wong 2011 色盲友好方案
5. Nature 单栏标准：3.5 inch 宽，300dpi，8pt字号
6. 去掉顶部和右侧边框
```

**预期改进**：图表从"能看"升级为"能发表"，所有科研规范覆盖。

### Round 3 — Inline Edit 微调

**微调1**：`Cmd+K` → "把标注箭头的 xytext 向左偏移 5 个单位，避免遮挡数据点"

**微调2**：`Cmd+K` → "置信区间的 alpha 从 0.15 改为 0.12，减少视觉干扰"

**最终效果**：微调后标注不再遮挡数据，置信区间更加subtle。

### Round 1 vs Round 3 对比

| 维度 | Round 1 | Round 3 |
|------|---------|---------|
| 信息传达 | 只有趋势线，缺乏统计证据 | 趋势+CI+统计检验+最暖年标注 |
| 视觉美观 | 默认配色，字号不统一 | Wong配色，统一8pt，简洁 |
| 科研规范 | 不满足任何发表要求 | 完全符合Nature标准 |
| 代码简洁 | 10行 | 40行（但结构清晰） |

### Prompt 设计策略心得

三轮迭代的核心策略是"从模糊到精确"的渐进收敛：Round 1 用自然语言描述"感觉"，让 AI 理解大方向；Round 2 用结构化约束条件逐条收紧，确保每个细节符合标准；Round 3 用 Inline Edit 做外科手术式微调，处理只有目视才能发现的细微问题。关键认知：不要试图在 Round 1 就把所有要求写全——信息过载会让 AI 遗漏重要约束，不如分层递进。

## 思考题参考答案

### 1. AI 生成的配色是否色盲友好？如何验证？

AI 默认配色往往不是色盲友好的（常用红绿对比）。验证方法：
- 使用 colorblindcheck 工具在线模拟（https://www.color-blindness.com/coblis-color-blindness-simulator/）
- 将图表转为灰度，检查各系列是否仍可区分
- 参照 Wong 2011 推荐的7色方案

### 2. 误差棒 vs 置信区间，各适合什么场景？

- **误差棒（errorbar）**：适合离散数据点（如各年代的均值±标准差），强调每个点的不确定性
- **置信区间（fill_between）**：适合连续数据（如时间序列的滑动均值），强调趋势的可信范围
- 简单规则：离散用点+误差棒，连续用线+带状填充

### 3. 双轴图(twinx)为什么容易产生误导？

- 两个 Y 轴的范围可以任意缩放，通过调整范围可以让无关的两条线看起来"相关"
- 读者容易误解为两个变量有因果关系
- 避免方法：1) 标注清楚两个轴的含义和单位 2) 不用于暗示因果 3) 优先用归一化后的单轴图